In [ ]:
# ========== SETUP & INSTALL ==========
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q albumentations opencv-python-headless scikit-learn pandas matplotlib seaborn kaggle

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ========== KAGGLE API SETUP ==========
import os
import json
from pathlib import Path

print("=== Setting up Kaggle API ===")

# Use credentials provided
KAGGLE_USERNAME = "kiramada"
KAGGLE_KEY = "985c4b29a5b9f8ac99ff910845c5521c"

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

# Create kaggle directory and file
kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / 'kaggle.json'
with open(kaggle_json, 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(kaggle_json, 0o600)
print("✓ Kaggle credentials set")

# Verify
!kaggle datasets list -s beach-sand-mineral 2>&1 | head -2
print("✓ Kaggle API configured successfully!")

In [ ]:
# ========== DOWNLOAD DATASETS ==========
import subprocess
import zipfile
from pathlib import Path

print("=== Downloading Datasets ===")

# Create data directories
base_dir = Path('/content/BSM_Project_3')
base_dir.mkdir(parents=True, exist_ok=True)
combined_dir = Path('/content/combined_dataset')
combined_dir.mkdir(parents=True, exist_ok=True)
split_dir = Path('/content/data_info')
split_dir.mkdir(parents=True, exist_ok=True)

# Download split dataset
print("Downloading split dataset...")
!kaggle datasets download kiramada/bsm-dataset-splits -p /content/data_info --unzip -q

# Download combined dataset
print("Downloading combined dataset...")
!kaggle datasets download kiramada/bsm-combined-augmented -p /content --unzip -q

# Download original dataset (optional backup)
print("Downloading original dataset...")
!kaggle datasets download prasannavenkatesant/beach-sand-mineral-bsm -p /content --unzip -q

print("\n=== Download Complete ===")

In [ ]:
# ========== VERIFY DOWNLOADED DATA ==========
from pathlib import Path
import os

print("=== Verifying Downloaded Data ===")

# Check split file
split_path = Path('/content/data_info/primary_split.json')
print(f"Split file exists: {Path('/content/data_info/primary_split.json').exists()}")

# Check combined dataset
combined_dir = Path('/content/combined_dataset')
print(f"\nCombined dataset:")
if Path('/content/combined_dataset').exists():
    for cls_dir in sorted(Path('/content/combined_dataset').iterdir()):
        if cls_dir.is_dir():
            count = len(list(cls_dir.glob('*.jpg')))
            print(f"  {cls_dir.name}: {count} images")
            # Check first few files
            for img in list(cls_dir.glob('*.jpg'))[:3]:
                print(f"    Example: {img.name} (exists: {img.exists()})")
else:
    print(f"  Combined dir not found at /content/combined_dataset")
    print(f"  Checking alternative locations...")
    for p in [
        '/content/combined_dataset',
        '/content/bsm-combined-augmented',
        '/content/bsm-combined-augmented',
    ]:
        if Path(p).exists():
            print(f"  Found at: {p}")
            for cls_dir in sorted(Path(p).iterdir()):
                if cls_dir.is_dir():
                    count = len(list(cls_dir.glob('*.jpg')))
                    print(f"  {cls_dir.name}: {len(list(cls_dir.glob('*.jpg')))} images")

# Check original dataset
orig_path = Path('/content/beach-sand-mineral-bsm/BSM')
print(f"\nOriginal dataset exists: {orig_path.exists()}")
if orig_path.exists():
    for cls_dir in sorted(orig_path.iterdir()):
        if cls_dir.is_dir():
            count = len(list(cls_dir.glob('*.jpg')))
            print(f"  {cls_dir.name}: {count} images")

In [ ]:
# ========== EVALUATE ALL 4 MODELS ==========
import torch, json, cv2, os
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms as T
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

IMG_SIZE = 224
BATCH_SIZE = 32
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Load split
with open('/content/data_info/primary_split.json') as f:
    split_data = json.load(f)
test_data = split_data['test']

CLASS_NAMES = ['Garnet', 'Ilmenite', 'Monazite', 'Rutile', 'Sillimanite', 'Zircon']
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASS_NAMES)}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
VAL_TEST_TRANSFORM = T.Compose([
    T.ToPILImage(),
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class BSMDataset(Dataset):
    def __init__(self, data_list, transform=None):
        self.data_list = data_list
        self.transform = transform
    def __len__(self):
        return len(self.data_list)
    def __getitem__(self, idx):
        item = self.data_list[idx]
        img_path = item['path']
        label = CLASS_TO_IDX[item['class']]
        image = cv2.imread(img_path)
        if image is None:
            raise ValueError(f'Failed to load image: {img_path}')
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image)
        return image, label

test_dataset = BSMDataset(test_data, transform=VAL_TEST_TRANSFORM)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

os.makedirs('/content/results/metrics', exist_ok=True)
os.makedirs('/content/results/confusion_matrices', exist_ok=True)
os.makedirs('/content/results/curves', exist_ok=True)

def evaluate_model(model_name, ckpt_path):
    print(f'\n=== Evaluating {model_name} ===')
    if model_name == 'resnet18':
        model = models.resnet18(weights=None)
        model.fc = torch.nn.Linear(model.fc.in_features, 6)
    elif model_name == 'resnet34':
        model = models.resnet34(weights=None)
        model.fc = torch.nn.Linear(model.fc.in_features, 6)
    elif model_name == 'dresnet50':
        model = models.resnet50(weights=None)
        model.fc = torch.nn.Linear(model.fc.in_features, 6)
    elif model_name == 'vgg16':
        model = models.vgg16(weights=None)
        model.classifier[6] = torch.nn.Linear(4096, 6)
    else:
        raise ValueError(model_name)
    
    ckpt = torch.load(ckpt_path, map_location=DEVICE)
    if isinstance(ckpt, dict) and 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
        print(f'Loaded from checkpoint (epoch {ckpt.get("epoch", "N/A")}, val_acc {ckpt.get("val_acc", "N/A")})')
    else:
        model.load_state_dict(ckpt)
    
    model = model.to(DEVICE)
    model.eval()
    
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='weighted')
    print(f'Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}')
    
    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True)
    with open(f'/content/results/metrics/{model_name}_test_metrics.json', 'w') as f:
        json.dump(report, f, indent=2)
    df = pd.DataFrame(report).transpose()
    df.to_csv(f'/content/results/metrics/{model_name}_test_metrics.csv')
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap='Blues')
    plt.title(f'{model_name} Confusion Matrix (Test)')
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    plt.savefig(f'/content/results/confusion_matrices/{model_name}_confusion_matrix.png', dpi=300)
    plt.close()
    
    return {'model': model_name, 'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

results = []
for mname in ['resnet18', 'resnet34', 'dresnet50', 'vgg16']:
    for p in [f'/content/{mname}_best.pth', f'/content/{mname}_epoch60.pth', f'/content/{mname}.pth']:
        if os.path.exists(p):
            results.append(evaluate_model(mname, p))
            break
    else:
        print(f'No checkpoint found for {mname}')

with open('/content/results/metrics/all_models_test_results.json', 'w') as f:
    json.dump(results, f, indent=2)

df = pd.DataFrame(results)
df.to_csv('/content/results/metrics/all_models_test_results.csv', index=False)
print('\n=== ALL RESULTS ===')
print(df.to_string(index=False))
df.to_markdown('/content/results/metrics/all_models_comparison.md', index=False)
print('\nSaved to /content/results/')

In [ ]:
# ========== DOWNLOAD RESULTS ==========
from google.colab import files
import os

print("=== Packaging results for download ===")

# Create zip of all results
!cd /content && zip -r bsm_results.zip results/

# Download
files.download('/content/bsm_results.zip')

print("Done! Download the zip file and extract to C:/BSM Project 3/results/")